In [ ]:
import gc
import importlib
import inspect
import json
import re
import sqlite3
import sys
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch

# STEP 1: CHECK T4 GPU, INSTALL LIBRARIES & EXTRACT DATA
if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab is running on CPU! Go to Runtime -> Change runtime type -> select T4 GPU and run again."
    )

zip_path = Path("eval_for_colab.zip")
if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    raise RuntimeError("Valid 'eval_for_colab.zip' file not found. Please upload and wait for 100% completion!")

print(f">>> GPU detected: {torch.cuda.get_device_name(0)}. Installing libraries...")
!pip install -q chromadb rank-bm25 pyyaml langchain-community > /dev/null 2>&1

!rm -rf "Evaluation" "Retrieval" "Query Understanding and Routing" "Document Ingestion and Parsing" eval_results.zip
!unzip -q -o eval_for_colab.zip

for req_dir in (
    "Document Ingestion and Parsing/output/Ingestion",
    "Document Ingestion and Parsing/output/Chunking",
    "Document Ingestion and Parsing/output/Provenance",
    "Document Ingestion and Parsing/output/Vector DB",
):
    if not Path(req_dir).is_dir():
        raise FileNotFoundError(
            f"Missing directory '{req_dir}' in the archive. Please re-run pack_eval.py on your computer!"
        )

ROOT_DIR = Path.cwd()
RETRIEVAL_DIR = ROOT_DIR / "Retrieval"
for p in (ROOT_DIR, RETRIEVAL_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from path import CHUNKING_OUT_DIR, PROVENANCE_OUT_DIR, STRATEGY_FOLDER_MAP
from sentence_transformers import CrossEncoder, SentenceTransformer

EVAL_DIR = Path("Evaluation/Chunking Evals")
REPORT_L1 = EVAL_DIR / "eval_layer1_report.json"
REPORT_L2 = EVAL_DIR / "eval_layer2_report.json"

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?。！？])\s+|\n+")
HEADING_PREFIX_RE = re.compile(r"^\[[^\]]+\]\n?")
STRATEGIES = ["fixed_size", "structure_aware", "semantic"]
ATOMIC_BLOCK_TYPES = {"table", "code", "record"}

# STEP 2: LAYER 1 · INTRINSIC EVALUATION (ALL 48 DOCUMENTS)
print("\n>>> [Layer 1] Initializing BAAI/bge-m3 (FP16) on CUDA...")
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda")
embedder.max_seq_length = 256
embedder.half()

report_l1 = {}
for slug in STRATEGIES:
    print(f">>> [Layer 1 - Intrinsic] Evaluating: {slug} ...")
    folder_name = STRATEGY_FOLDER_MAP.get(slug, slug)
    db_path = PROVENANCE_OUT_DIR / slug / "workspace_data.db"
    chunk_dir = CHUNKING_OUT_DIR / folder_name

    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row

    block_meta = {
        r["normalized_id"]: (
            r["block_type"],
            r["heading_path_json"] or "[]",
            len(TOKEN_RE.findall(r["normalized_text"] or "")),
        )
        for r in conn.execute(
            "SELECT normalized_id, block_type, heading_path_json, normalized_text FROM normalized_blocks"
        )
    }

    chunk_blocks = defaultdict(list)
    block_to_chunks = defaultdict(set)
    for r in conn.execute("SELECT chunk_id, normalized_block_id FROM chunk_sources"):
        chunk_blocks[r["chunk_id"]].append(r["normalized_block_id"])
        block_to_chunks[r["normalized_block_id"]].add(r["chunk_id"])
    conn.close()

    eligible = [b for b, (bt, _, tlen) in block_meta.items() if bt in ATOMIC_BLOCK_TYPES and 0 < tlen <= 650]
    intact = sum(1 for b in eligible if len(block_to_chunks.get(b, set())) == 1)
    boundary_integrity = (intact / len(eligible)) if eligible else 0.0

    pure_chunks = sum(
        1 for bids in chunk_blocks.values()
        if len({block_meta[b][1] for b in bids if b in block_meta}) == 1
    )
    heading_purity = (pure_chunks / len(chunk_blocks)) if chunk_blocks else 0.0

    tails, heads = [], []
    total_chunks, total_tokens = 0, 0
    for jf in sorted(chunk_dir.glob("*.json")):
        chunks = json.loads(jf.read_text(encoding="utf-8")).get("chunks", [])
        chunks.sort(key=lambda c: c.get("chunk_idx", 0))
        total_chunks += len(chunks)
        total_tokens += sum(c.get("token_count", 0) for c in chunks)

        for i in range(len(chunks) - 1):
            c1, c2 = chunks[i], chunks[i + 1]
            if not c1.get("heading_path") or c1.get("heading_path") != c2.get("heading_path"):
                continue
            b1 = HEADING_PREFIX_RE.sub("", c1.get("text_content", "")).strip()
            b2 = HEADING_PREFIX_RE.sub("", c2.get("text_content", "")).strip()
            s1 = [s.strip() for s in SENTENCE_SPLIT_RE.split(b1) if s.strip()]
            s2 = [s.strip() for s in SENTENCE_SPLIT_RE.split(b2) if s.strip()]
            if s1 and s2:
                tails.append(s1[-1][:512])
                heads.append(s2[0][:512])

    continuity = 0.0
    if tails:
        with torch.inference_mode():
            v_tails = embedder.encode(tails, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
            v_heads = embedder.encode(heads, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
        continuity = float(np.mean(np.sum(v_tails * v_heads, axis=1)))

    report_l1[slug] = {
        "total_chunks": total_chunks,
        "avg_tokens": round(total_tokens / total_chunks, 1) if total_chunks else 0.0,
        "eligible_atomic_blocks": len(eligible),
        "intact_atomic_blocks": intact,
        "boundary_integrity": round(boundary_integrity * 100, 2),
        "heading_purity": round(heading_purity * 100, 2),
        "boundary_continuity": round(continuity, 4),
    }

    del block_meta, chunk_blocks, block_to_chunks, tails, heads
    gc.collect()
    torch.cuda.empty_cache()

REPORT_L1.write_text(json.dumps(report_l1, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n=== LAYER 1 · INTRINSIC EVALUATION (ALL 48 DOCUMENTS) ===")
print("| Strategy | Chunks | Avg Tokens | Boundary Integrity | Heading Purity | Boundary Continuity |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")
for slug in STRATEGIES:
    m = report_l1[slug]
    print(
        f"| **{slug}** | {m['total_chunks']} | {m['avg_tokens']} | "
        f"{m['boundary_integrity']}% ({m['intact_atomic_blocks']}/{m['eligible_atomic_blocks']}) | "
        f"{m['heading_purity']}% | {m['boundary_continuity']} |"
    )

# Completely release the Layer 1 model before transitioning to Layer 2
del embedder
gc.collect()
torch.cuda.empty_cache()

# LAYER 2 - RETRIEVAL-BASED EVALUATION (CHROMA TOKEN-LEVEL @ TOP-5)
# Patch CrossEncoder.predict compatibility issue on sentence-transformers v5 + transformers v5
def _safe_ce_predict(self, sentences, batch_size: int = 32, show_progress_bar: bool = False, **kwargs):
    if not sentences:
        return []
    dev = next(self.model.parameters()).device
    max_len = getattr(self, "max_length", None) or 512
    all_scores: list[float] = []
    self.model.eval()
    with torch.inference_mode():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i : i + batch_size]
            enc = self.tokenizer(
                [str(p[0]) for p in batch],
                [str(p[1]) for p in batch],
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            enc = {k: v.to(dev) for k, v in enc.items()}
            logits = self.model(**enc).logits.view(-1)
            all_scores.extend(torch.sigmoid(logits).float().cpu().tolist())
    return all_scores

CrossEncoder.predict = _safe_ce_predict

import query_vectorizer
import reranker
import retrieval_system
importlib.reload(query_vectorizer)
importlib.reload(reranker)
importlib.reload(retrieval_system)

from query_vectorizer import QueryVectorizer
from reranker import CrossEncoderReranker
from retrieval_system import RetrievalSystem

# Force QueryVectorizer to run on CUDA FP16 + vector cache for MMR
_orig_qv_get = QueryVectorizer._get_model
def _cuda_qv_get(self):
    self.device = "cuda"
    m = _orig_qv_get(self).to("cuda")
    if not getattr(m, "_is_fp16", False):
        m.half()
        m._is_fp16 = True
    return m

_passage_cache: dict[str, list[float]] = {}
_orig_vec_passages = QueryVectorizer.vectorize_passages
def _cached_vec_passages(self, texts: list[str]) -> list[list[float]]:
    clean = [(t or "").strip() for t in texts]
    missing = [t for t in dict.fromkeys(clean) if t and t not in _passage_cache]
    if missing:
        with torch.inference_mode():
            encoded = _orig_vec_passages(self, missing)
        for txt, vec in zip(missing, encoded):
            _passage_cache[txt] = vec
    return [_passage_cache.get(t, []) if t else [] for t in clean]

QueryVectorizer._get_model = _cuda_qv_get
QueryVectorizer.vectorize_passages = _cached_vec_passages

# Force CrossEncoderReranker to run on CUDA FP16
_orig_ce_get = CrossEncoderReranker._get_model
def _cuda_ce_get(self):
    self.device = "cuda"
    ce = _orig_ce_get(self)
    ce.to("cuda")
    if hasattr(ce, "model") and not getattr(ce.model, "_is_fp16", False):
        ce.model.half()
        ce.model._is_fp16 = True
    return ce

CrossEncoderReranker._get_model = _cuda_ce_get

# Initialize 1 Vectorizer + 1 Reranker (threshold=0.0 to avoid blocking VI-EN queries) shared across all 3 strategies
vec_kwargs = {"device": "cuda"} if "device" in inspect.signature(QueryVectorizer.__init__).parameters else {}
vectorizer = QueryVectorizer(**vec_kwargs)

rerank_kwargs = {"threshold": 0.0, "top_k": None}
if "device" in inspect.signature(CrossEncoderReranker.__init__).parameters:
    rerank_kwargs["device"] = "cuda"
shared_reranker = CrossEncoderReranker(**rerank_kwargs)

qv_dev = next(vectorizer._get_model().parameters()).device
ce_dev = next(shared_reranker._get_model().model.parameters()).device
print(f"\n>>> [Layer 2] Execution devices: QueryVectorizer={qv_dev}, CrossEncoderReranker={ce_dev}")


def merge_intervals(intervals: list[tuple[int, int]]) -> list[tuple[int, int]]:
    valid = sorted((s, e) for s, e in intervals if e > s)
    if not valid:
        return []
    merged = [valid[0]]
    for s, e in valid[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged


def intervals_to_token_set(
    ver_id: str,
    doc_token_spans: list[tuple[int, int]],
    merged_intervals: list[tuple[int, int]],
) -> set[tuple[str, int]]:
    tokens: set[tuple[str, int]] = set()
    if not merged_intervals or not doc_token_spans:
        return tokens
    i_idx, n_int = 0, len(merged_intervals)
    for t_idx, (ts, te) in enumerate(doc_token_spans):
        while i_idx < n_int and merged_intervals[i_idx][1] <= ts:
            i_idx += 1
        if i_idx >= n_int:
            break
        ms, me = merged_intervals[i_idx]
        if max(ts, ms) < min(te, me):
            tokens.add((ver_id, t_idx))
    return tokens


def build_coord_index(slug: str):
    db_path = PROVENANCE_OUT_DIR / slug / "workspace_data.db"
    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row

    blocks_by_ver = defaultdict(list)
    for r in conn.execute(
        "SELECT normalized_id, version_id, block_idx, normalized_text FROM normalized_blocks ORDER BY version_id, block_idx ASC"
    ):
        blocks_by_ver[r["version_id"]].append(r)

    block_doc_start, block_ver_map, doc_token_spans = {}, {}, {}
    for ver_id, v_blocks in blocks_by_ver.items():
        parts, cursor = [], 0
        for idx, r in enumerate(v_blocks):
            if idx > 0:
                parts.append("\n\n")
                cursor += 2
            t = r["normalized_text"] or ""
            block_doc_start[r["normalized_id"]] = cursor
            block_ver_map[r["normalized_id"]] = ver_id
            parts.append(t)
            cursor += len(t)
        doc_token_spans[ver_id] = [(m.start(), m.end()) for m in TOKEN_RE.finditer("".join(parts))]

    raw_ivs = defaultdict(lambda: defaultdict(list))
    for r in conn.execute(
        "SELECT chunk_id, normalized_block_id, start_offset, end_offset FROM chunk_sources ORDER BY chunk_id, source_order"
    ):
        bid = r["normalized_block_id"]
        if bid in block_doc_start:
            base = block_doc_start[bid]
            raw_ivs[r["chunk_id"]][block_ver_map[bid]].append((base + r["start_offset"], base + r["end_offset"]))
    conn.close()

    chunk_tokens = {}
    for cid, vmap in raw_ivs.items():
        ct = set()
        for vid, ivs in vmap.items():
            ct |= intervals_to_token_set(vid, doc_token_spans[vid], merge_intervals(ivs))
        chunk_tokens[cid] = ct
    return block_doc_start, doc_token_spans, chunk_tokens


gt_candidates = sorted(EVAL_DIR.glob("ground_truth*.json"))
if not gt_candidates:
    raise FileNotFoundError(f"No ground_truth*.json file found in {EVAL_DIR}")
gt_path = EVAL_DIR / "ground_truth_chunking.json" if (EVAL_DIR / "ground_truth_chunking.json").exists() else gt_candidates[0]
print(f">>> Using Ground Truth: {gt_path.name}")

gt_units = json.loads(gt_path.read_text(encoding="utf-8"))
report_l2 = {}

for slug in STRATEGIES:
    t0 = time.perf_counter()
    print(f"\n>>> [Layer 2 - Retrieval] Evaluating strategy: {slug} ...")
    b_start, doc_spans, c_toks = build_coord_index(slug)
    rs = RetrievalSystem(query_vectorizer=vectorizer, chunking_strategy=slug, reranker=shared_reranker)

    overall = defaultdict(list)
    by_cat = defaultdict(lambda: defaultdict(list))
    by_lang = defaultdict(lambda: defaultdict(list))

    for idx, unit in enumerate(gt_units, 1):
        spans = unit.get("spans") or []
        if not spans:
            continue
        query = (unit.get("query") or "").strip()
        cat, lang, ver_id = unit.get("category", "unknown"), unit.get("lang", "unknown"), unit["version_id"]

        raw_g = [
            (sp["doc_char_start"], sp["doc_char_end"])
            if "doc_char_start" in sp
            else (
                b_start[sp["normalized_block_id"]] + sp["start_offset"],
                b_start[sp["normalized_block_id"]] + sp["end_offset"],
            )
            for sp in spans
        ]
        t_gold = intervals_to_token_set(ver_id, doc_spans.get(ver_id, []), merge_intervals(raw_g))
        if not t_gold:
            continue

        plan = {
            "resolved_query": query,
            "original_query": query,
            "intent": "SEARCH",
            "confidence": 1.0,
            "context_used": False,
            "entities": {"document": [], "vulnerabilities": []},
            "filter": {"version": None, "document_type": None, "source": []},
            "task": [
                {
                    "task_id": "eval_task_001",
                    "intent_task": "SEARCH",
                    "query": query,
                    "objective": {"type": "vulnerabilities", "ids": []},
                    "depends_on": None,
                }
            ],
            "need_clarify": False,
            "clarification_message": None,
        }
        out = rs.retrieve(plan)
        cids = [ev["chunk_id"] for ev in out.get("evidence_list", [])[:5]]
        t_ret = set()
        for cid in cids:
            t_ret |= c_toks.get(cid, set())

        if not t_ret:
            rec = prec = iou = 0.0
        else:
            inter = len(t_ret & t_gold)
            rec = inter / len(t_gold)
            prec = inter / len(t_ret)
            iou = inter / len(t_ret | t_gold)

        for k, v in (("token_recall", rec), ("token_precision", prec), ("token_iou", iou)):
            overall[k].append(v)
            by_cat[cat][k].append(v)
            by_lang[lang][k].append(v)

        if idx % 10 == 0 or idx == len(gt_units):
            print(f"   ... [{slug}] {idx}/{len(gt_units)} queries ({time.perf_counter() - t0:.1f}s)")

    def _sum(bucket):
        return {
            k: {
                "token_recall@5": round(float(np.mean(v["token_recall"])) * 100, 2),
                "token_precision@5": round(float(np.mean(v["token_precision"])) * 100, 2),
                "token_iou@5": round(float(np.mean(v["token_iou"])) * 100, 2),
            }
            for k, v in bucket.items()
        }

    report_l2[slug] = {
        "token_recall@5": round(float(np.mean(overall["token_recall"])) * 100, 2),
        "token_precision@5": round(float(np.mean(overall["token_precision"])) * 100, 2),
        "token_iou@5": round(float(np.mean(overall["token_iou"])) * 100, 2),
        "by_category": _sum(by_cat),
        "by_lang": _sum(by_lang),
    }

REPORT_L2.write_text(json.dumps(report_l2, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n=== LAYER 2 · RETRIEVAL-BASED EVALUATION (CHROMA TOKEN-LEVEL @ TOP-5) ===")
print("| Strategy | Token Recall@5 | Token Precision@5 | Token IoU@5 |")
print("| :--- | :---: | :---: | :---: |")
for slug in STRATEGIES:
    m = report_l2[slug]
    print(f"| **{slug}** | {m['token_recall@5']}% | {m['token_precision@5']}% | **{m['token_iou@5']}%** |")

print("\n=== LAYER 2 · BREAKDOWN BY QUERY LANGUAGE (Recall@5 / Precision@5 / IoU@5) ===")
print("| Language | Fixed-size | Structure-aware | Semantic |")
print("| :--- | :---: | :---: | :---: |")
for lang in ("en", "vi"):
    cells = [
        f"{report_l2[s]['by_lang'].get(lang, {}).get('token_recall@5', 0)}% / "
        f"{report_l2[s]['by_lang'].get(lang, {}).get('token_precision@5', 0)}% / "
        f"**{report_l2[s]['by_lang'].get(lang, {}).get('token_iou@5', 0)}%**"
        for s in STRATEGIES
    ]
    print(f"| **{lang.upper()}** | {cells[0]} | {cells[1]} | {cells[2]} |")

print("\n=== LAYER 2 · BREAKDOWN BY 4 SCENARIO GROUPS (Recall@5 / Precision@5 / IoU@5) ===")
print("| Category | Fixed-size | Structure-aware | Semantic |")
print("| :--- | :---: | :---: | :---: |")
for cat in ["prose_topic", "atomic_structure", "heading_plus_body", "multi_block_rule"]:
    cells = [
        f"{report_l2[s]['by_category'].get(cat, {}).get('token_recall@5', 0)}% / "
        f"{report_l2[s]['by_category'].get(cat, {}).get('token_precision@5', 0)}% / "
        f"**{report_l2[s]['by_category'].get(cat, {}).get('token_iou@5', 0)}%**"
        for s in STRATEGIES
    ]
    print(f"| **{cat}** | {cells[0]} | {cells[1]} | {cells[2]} |")

# COMPRESS ALL JSON RESULTS AND DOWNLOAD TO LAPTOP
!zip -q -r eval_results.zip "Evaluation/Chunking Evals" -x "*__pycache__*"
from google.colab import files
files.download("eval_results.zip")